# Speech Audio Denoiser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szymon-Jablonski/ml-audio-denoiser/blob/main/audio_denoiser.ipynb)

University ML project: train a PyTorch model that takes a noisy speech recording and produces a cleaner version.

**Before running:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom.

Pipeline:
1. Time domain → frequency domain (STFT / Mel spectrogram)
2. Denoise in the frequency domain
3. Go back to the time domain (inverse STFT)

## 1. Environment setup

In [ ]:
# Colab already ships with torch and torchaudio; this makes sure the remaining packages are present.
!pip install -q soundfile matplotlib

In [ ]:
import os
import random
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}, torchaudio {torchaudio.__version__}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found - in Colab use Runtime -> Change runtime type -> T4 GPU")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Configuration and storage

Colab's local disk is wiped when the runtime disconnects. Set `USE_DRIVE = True` to keep datasets and checkpoints in Google Drive.

In [ ]:
USE_DRIVE = False

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/ml-audio-denoiser"
elif IN_COLAB:
    PROJECT_DIR = "/content/ml-audio-denoiser"
else:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SAMPLE_RATE = 16_000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64

print("Data dir:      ", DATA_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)

## 3. Data

**TODO:** load a real paired dataset (e.g. VoiceBank-DEMAND: noisy speech + matching clean speech).

For now a short demo speech clip is downloaded and white noise is added at a chosen SNR to create a `(noisy, clean)` pair.

In [ ]:
SAMPLE_URL = "https://download.pytorch.org/torchaudio/tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
sample_path = os.path.join(DATA_DIR, "sample_speech.wav")
if not os.path.exists(sample_path):
    urllib.request.urlretrieve(SAMPLE_URL, sample_path)


def load_audio(path, sample_rate=SAMPLE_RATE):
    """Load an audio file as a mono float tensor of shape (1, samples), resampled to `sample_rate`."""
    data, sr = sf.read(path, dtype="float32", always_2d=True)  # (samples, channels)
    wav = torch.from_numpy(data.T).mean(dim=0, keepdim=True)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sr, sample_rate)
    return wav


def add_noise(clean, snr_db):
    """Add white Gaussian noise to `clean` at the given signal-to-noise ratio (dB)."""
    noise = torch.randn_like(clean)
    signal_power = clean.pow(2).mean()
    noise_power = noise.pow(2).mean()
    scale = torch.sqrt(signal_power / (noise_power * 10 ** (snr_db / 10)))
    return clean + scale * noise


clean = load_audio(sample_path)
noisy = add_noise(clean, snr_db=5)
print(f"Clip length: {clean.shape[-1] / SAMPLE_RATE:.2f} s")

## 4. Signal processing helpers

In [ ]:
window = torch.hann_window(N_FFT)


def stft(wav):
    """Time domain -> complex spectrogram (1, freq, frames)."""
    return torch.stft(wav, n_fft=N_FFT, hop_length=HOP_LENGTH,
                      window=window.to(wav.device), return_complex=True)


def istft(spec, length=None):
    """Complex spectrogram -> time domain."""
    return torch.istft(spec, n_fft=N_FFT, hop_length=HOP_LENGTH,
                       window=window.to(spec.device), length=length)


mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)


def plot_audio(wav, title):
    """Plot the waveform and the log-magnitude spectrogram."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    t = np.arange(wav.shape[-1]) / SAMPLE_RATE
    axes[0].plot(t, wav[0].cpu().numpy(), linewidth=0.5)
    axes[0].set(title=f"{title} - waveform", xlabel="Time [s]")
    mag_db = 20 * torch.log10(stft(wav.cpu()).abs() + 1e-8)
    axes[1].imshow(mag_db[0].numpy(), origin="lower", aspect="auto",
                   extent=[0, t[-1], 0, SAMPLE_RATE / 2])
    axes[1].set(title=f"{title} - spectrogram", xlabel="Time [s]", ylabel="Frequency [Hz]")
    plt.tight_layout()
    plt.show()


def play(wav):
    display(Audio(wav[0].cpu().numpy(), rate=SAMPLE_RATE))

In [ ]:
plot_audio(clean, "Clean")
play(clean)
plot_audio(noisy, "Noisy (5 dB SNR)")
play(noisy)

# Sanity check: STFT -> iSTFT reconstructs the signal
reconstructed = istft(stft(noisy), length=noisy.shape[-1])
print("Reconstruction error:", (reconstructed - noisy).abs().max().item())

## 5. Model

**TODO:** define the denoising model. A common starting point is predicting a mask on the magnitude spectrogram
(e.g. a small CNN / U-Net), multiplying it with the noisy spectrogram and converting back with the noisy phase.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: replace with the real architecture
        self.net = nn.Identity()

    def forward(self, noisy_mag):
        return self.net(noisy_mag)


model = Denoiser().to(device)
print(model)

## 6. Training

**TODO:** dataset/dataloader, loss function, optimizer and training loop. Save checkpoints to `CHECKPOINT_DIR`.

In [ ]:
# TODO: training loop
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# for epoch in range(num_epochs):
#     ...
# torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "denoiser.pt"))

## 7. Evaluation

In [ ]:
def snr(reference, estimate):
    """Signal-to-noise ratio of `estimate` with respect to `reference`, in dB."""
    noise = reference - estimate
    return 10 * torch.log10(reference.pow(2).sum() / (noise.pow(2).sum() + 1e-8)).item()


@torch.no_grad()
def denoise(wav):
    """Run the model on a waveform: STFT -> model on magnitude -> iSTFT with the noisy phase."""
    model.eval()
    spec = stft(wav.to(device))
    mag, phase = spec.abs(), torch.angle(spec)
    est_mag = model(mag)
    return istft(torch.polar(est_mag, phase), length=wav.shape[-1]).cpu()


denoised = denoise(noisy)
print(f"SNR noisy:    {snr(clean, noisy):6.2f} dB")
print(f"SNR denoised: {snr(clean, denoised):6.2f} dB")
plot_audio(denoised, "Denoised")
play(denoised)

## 8. Results and conclusions

**TODO:** compare with other denoising methods (e.g. spectral subtraction, Wiener filter), summarise listening tests,
and describe where the model works well and where it struggles.